In [1]:
import os
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'  # 用于调试CUDA错误
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'max_split_size_mb:128'
import torch
import torch.nn as nn
import torch.nn.functional as F
import dgl
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
import scanpy as sc
import anndata
import matplotlib.pyplot as plt
import seaborn as sns

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

Using device: cuda


In [2]:
def zscore_norm(x):
    m = x.mean(dim=0, keepdim=True)
    s = x.std(dim=0, keepdim=True) + 1e-8
    return (x - m) / s

def create_heterogeneous_graph():
    adj_real_pseudo = pd.read_csv('./adj_real_pseudo.csv', index_col=0)
    adj_real_real = pd.read_csv('./adj_real_real.csv', index_col=0)
    adj_realspot_gene = pd.read_csv('./adj_realspot_gene.csv', index_col=0)
    adj_pseuspot_gene = pd.read_csv('./adj_pseuspot_gene.csv', index_col=0)

    ei_rp = torch.tensor(np.array(np.where(adj_real_pseudo.values > 0)), dtype=torch.long)
    ei_rr = torch.tensor(np.array(np.where(adj_real_real.values > 0)), dtype=torch.long)
    ei_rg = torch.tensor(np.array(np.where(adj_realspot_gene.values > 0)), dtype=torch.long)
    ei_pg = torch.tensor(np.array(np.where(adj_pseuspot_gene.values > 0)), dtype=torch.long)

    ew_rp = torch.tensor(adj_real_pseudo.values[adj_real_pseudo.values > 0], dtype=torch.float32, device=device)
    ew_rr = torch.tensor(adj_real_real.values[adj_real_real.values > 0], dtype=torch.float32, device=device)
    ew_rg = torch.tensor(adj_realspot_gene.values[adj_realspot_gene.values > 0], dtype=torch.float32, device=device)
    ew_pg = torch.tensor(adj_pseuspot_gene.values[adj_pseuspot_gene.values > 0], dtype=torch.float32, device=device)

    ei_gr = ei_rg[[1, 0]]
    ei_gp = ei_pg[[1, 0]]
    ew_gr = ew_rg
    ew_gp = ew_pg

    g = dgl.heterograph({
        ('real', 'to_pseudo', 'pseudo'): (ei_rp[0], ei_rp[1]),
        ('real', 'to_real', 'real'): (ei_rr[0], ei_rr[1]),
        ('real', 'expresses_gene', 'gene'): (ei_rg[0], ei_rg[1]),
        ('pseudo', 'expresses_gene', 'gene'): (ei_pg[0], ei_pg[1]),
        ('gene', 'expressed_in_real', 'real'): (ei_gr[0], ei_gr[1]),
        ('gene', 'expressed_in_pseudo', 'pseudo'): (ei_gp[0], ei_gp[1]),
    }).to(device)

    g.nodes['real'].data['feat'] = real_spot_feats
    g.nodes['pseudo'].data['feat'] = pseudo_spot_feats
    g.nodes['gene'].data['feat'] = gene_feats

    g.edges['real', 'to_pseudo', 'pseudo'].data['weight'] = ew_rp
    g.edges['real', 'to_real', 'real'].data['weight'] = ew_rr
    g.edges['real', 'expresses_gene', 'gene'].data['weight'] = ew_rg
    g.edges['pseudo', 'expresses_gene', 'gene'].data['weight'] = ew_pg
    g.edges['gene', 'expressed_in_real', 'real'].data['weight'] = ew_gr
    g.edges['gene', 'expressed_in_pseudo', 'pseudo'].data['weight'] = ew_gp
    return g

print('辅助函数定义完成')

辅助函数定义完成


In [3]:
class HeteroGAT(nn.Module):
    def __init__(self, in_dim_dict, hidden_dim, num_cell_types, device, num_heads=4, dropout_rate=0.3):
        super().__init__()
        self.device = device
        self.num_heads = num_heads
        self.head_dim = hidden_dim // num_heads

        self.node_weights1 = nn.ParameterDict({
            'real':   nn.Parameter(torch.randn(in_dim_dict['real'],   hidden_dim, device=device)),
            'pseudo': nn.Parameter(torch.randn(in_dim_dict['pseudo'], hidden_dim, device=device)),
            'gene':   nn.Parameter(torch.randn(in_dim_dict['gene'],   hidden_dim, device=device)),
        })
        self.node_weights2 = nn.ParameterDict({
            'real':   nn.Parameter(torch.randn(hidden_dim, hidden_dim, device=device)),
            'pseudo': nn.Parameter(torch.randn(hidden_dim, hidden_dim, device=device)),
            'gene':   nn.Parameter(torch.randn(hidden_dim, hidden_dim, device=device)),
        })

        self.attn_params1 = nn.ParameterDict({
            'real_from_real':   nn.Parameter(torch.randn(num_heads, in_dim_dict['real']   + self.head_dim, device=device)),
            'real_from_pseudo': nn.Parameter(torch.randn(num_heads, in_dim_dict['real']   + self.head_dim, device=device)),
            'real_from_gene':   nn.Parameter(torch.randn(num_heads, in_dim_dict['real']   + self.head_dim, device=device)),
            'pseudo_from_real': nn.Parameter(torch.randn(num_heads, in_dim_dict['pseudo'] + self.head_dim, device=device)),
            'pseudo_from_gene': nn.Parameter(torch.randn(num_heads, in_dim_dict['pseudo'] + self.head_dim, device=device)),
            'gene_from_real':   nn.Parameter(torch.randn(num_heads, in_dim_dict['gene']   + self.head_dim, device=device)),
            'gene_from_pseudo': nn.Parameter(torch.randn(num_heads, in_dim_dict['gene']   + self.head_dim, device=device)),
        })
        self.attn_params2 = nn.ParameterDict({
            'real_from_real':   nn.Parameter(torch.randn(num_heads, 2*self.head_dim, device=device)),
            'real_from_pseudo': nn.Parameter(torch.randn(num_heads, 2*self.head_dim, device=device)),
            'real_from_gene':   nn.Parameter(torch.randn(num_heads, 2*self.head_dim, device=device)),
            'pseudo_from_real': nn.Parameter(torch.randn(num_heads, 2*self.head_dim, device=device)),
            'pseudo_from_gene': nn.Parameter(torch.randn(num_heads, 2*self.head_dim, device=device)),
            'gene_from_real':   nn.Parameter(torch.randn(num_heads, 2*self.head_dim, device=device)),
            'gene_from_pseudo': nn.Parameter(torch.randn(num_heads, 2*self.head_dim, device=device)),
        })

        self.node_biases = nn.ParameterDict({
            'real':   nn.Parameter(torch.zeros(hidden_dim, device=device)),
            'pseudo': nn.Parameter(torch.zeros(hidden_dim, device=device)),
            'gene':   nn.Parameter(torch.zeros(hidden_dim, device=device)),
        })

        self.dropout = nn.Dropout(dropout_rate)
        self.pseudo_out = nn.Linear(hidden_dim, num_cell_types)

        for n in self.node_weights1:
            nn.init.xavier_uniform_(self.node_weights1[n])
            nn.init.xavier_uniform_(self.node_weights2[n])
        for k in self.attn_params1:
            nn.init.xavier_uniform_(self.attn_params1[k])
        for k in self.attn_params2:
            nn.init.xavier_uniform_(self.attn_params2[k])
        nn.init.xavier_uniform_(self.pseudo_out.weight)
        self.pseudo_out.bias.data.zero_()

    def edge_type(self, src_type, dst_type):
        if src_type == 'real' and dst_type == 'real':   return 'to_real'
        if src_type == 'real' and dst_type == 'pseudo': return 'to_pseudo'
        if src_type == 'real' and dst_type == 'gene':   return 'expresses_gene'
        if src_type == 'pseudo' and dst_type == 'gene': return 'expresses_gene'
        if src_type == 'gene' and dst_type == 'real':   return 'expressed_in_real'
        if src_type == 'gene' and dst_type == 'pseudo': return 'expressed_in_pseudo'
        return None

    def attn_first_layer(self, graph, src_type, dst_type, h_src, h_dst, W_src, attn_param):
        Wh_src = (h_src @ W_src).view(-1, self.num_heads, self.head_dim)
        out, etype = [], self.edge_type(src_type, dst_type)
        for i in range(h_dst.shape[0]):
            if etype is None:
                out.append(torch.zeros(self.num_heads*self.head_dim, device=h_dst.device)); continue
            nbr = graph.predecessors(i, etype=(src_type, etype, dst_type))
            if len(nbr) == 0:
                out.append(torch.zeros(self.num_heads*self.head_dim, device=h_dst.device)); continue
            v = torch.full((len(nbr),), i, dtype=nbr.dtype, device=nbr.device)
            eids = graph.edge_ids(nbr, v, etype=(src_type, etype, dst_type))
            w = graph.edges[(src_type, etype, dst_type)].data['weight'][eids].to(Wh_src.device)
            multi = []
            for head in range(self.num_heads):
                h_dst_i = h_dst[i].unsqueeze(0).expand(len(nbr), -1)
                Wh_src_n = Wh_src[nbr, head, :]
                attn_in = torch.cat([h_dst_i, Wh_src_n], dim=1)
                e_ij = F.leaky_relu(attn_in @ attn_param[head].unsqueeze(-1)).squeeze(-1)
                e_ij = e_ij + 0.1 * torch.log(w + 1e-8)
                alpha = torch.softmax(e_ij, dim=0)
                agg = (alpha.unsqueeze(-1) * Wh_src_n).sum(dim=0)
                multi.append(agg)
            out.append(torch.cat(multi, dim=0))
        return torch.stack(out, dim=0)

    def attn_second_layer(self, graph, src_type, dst_type, h_src, h_dst, W_src, W_dst, attn_param):
        Wh_src = (h_src @ W_src).view(-1, self.num_heads, self.head_dim)
        Wh_dst = (h_dst @ W_dst).view(-1, self.num_heads, self.head_dim)
        out, etype = [], self.edge_type(src_type, dst_type)
        for i in range(h_dst.shape[0]):
            if etype is None:
                out.append(torch.zeros(self.num_heads*self.head_dim, device=h_dst.device)); continue
            nbr = graph.predecessors(i, etype=(src_type, etype, dst_type))
            if len(nbr) == 0:
                out.append(torch.zeros(self.num_heads*self.head_dim, device=h_dst.device)); continue
            v = torch.full((len(nbr),), i, dtype=nbr.dtype, device=nbr.device)
            eids = graph.edge_ids(nbr, v, etype=(src_type, etype, dst_type))
            w = graph.edges[(src_type, etype, dst_type)].data['weight'][eids].to(Wh_src.device)
            multi = []
            for head in range(self.num_heads):
                Wh_dst_i = Wh_dst[i, head, :].unsqueeze(0).expand(len(nbr), -1)
                Wh_src_n = Wh_src[nbr, head, :]
                attn_in = torch.cat([Wh_dst_i, Wh_src_n], dim=1)
                e_ij = F.leaky_relu(attn_in @ attn_param[head].unsqueeze(-1)).squeeze(-1)
                e_ij = e_ij + 0.1 * torch.log(w + 1e-8)
                alpha = torch.softmax(e_ij, dim=0)
                agg = (alpha.unsqueeze(-1) * Wh_src_n).sum(dim=0)
                multi.append(agg)
            out.append(torch.cat(multi, dim=0))
        return torch.stack(out, dim=0)

    def post(self, h_old, h_new):
        h = h_new + h_old
        h = F.layer_norm(h, h.shape[-1:])
        return F.gelu(h)

    def custom_update_layer1(self, graph, h_real, h_pseudo, h_gene):
        r_from_p = self.attn_first_layer(graph, 'pseudo','real', h_pseudo, h_real, self.node_weights1['pseudo'], self.attn_params1['real_from_pseudo'])
        r_from_r = self.attn_first_layer(graph, 'real',  'real', h_real,   h_real, self.node_weights1['real'],   self.attn_params1['real_from_real'])
        r_from_g = self.attn_first_layer(graph, 'gene',  'real', h_gene,   h_real, self.node_weights1['gene'],   self.attn_params1['real_from_gene'])
        r_self = h_real @ self.node_weights1['real']
        h_real_new = self.dropout(F.relu(r_self + r_from_p + r_from_r + r_from_g + self.node_biases['real']))
    
        p_from_r = self.attn_first_layer(graph, 'real', 'pseudo', h_real,   h_pseudo, self.node_weights1['real'],  self.attn_params1['pseudo_from_real'])
        p_from_g = self.attn_first_layer(graph, 'gene', 'pseudo', h_gene,   h_pseudo, self.node_weights1['gene'],  self.attn_params1['pseudo_from_gene'])
        p_self = h_pseudo @ self.node_weights1['pseudo']
        h_pseudo_new = self.dropout(F.relu(p_self + p_from_r + p_from_g + self.node_biases['pseudo']))
    
        g_from_p = self.attn_first_layer(graph, 'pseudo','gene', h_pseudo, h_gene, self.node_weights1['pseudo'], self.attn_params1['gene_from_pseudo'])
        g_from_r = self.attn_first_layer(graph, 'real',  'gene', h_real,   h_gene, self.node_weights1['real'],   self.attn_params1['gene_from_real'])
        g_self = h_gene @ self.node_weights1['gene']
        h_gene_new = self.dropout(F.relu(g_self + g_from_p + g_from_r + self.node_biases['gene']))
        
        return h_real_new, h_pseudo_new, h_gene_new
    
    def custom_update_layer2(self, graph, h_real, h_pseudo, h_gene):
        r_from_p = self.attn_second_layer(graph, 'pseudo','real', h_pseudo, h_real, self.node_weights2['pseudo'], self.node_weights2['real'], self.attn_params2['real_from_pseudo'])
        r_from_r = self.attn_second_layer(graph, 'real',  'real', h_real,   h_real, self.node_weights2['real'],   self.node_weights2['real'], self.attn_params2['real_from_real'])
        r_from_g = self.attn_second_layer(graph, 'gene',  'real', h_gene,   h_real, self.node_weights2['gene'],   self.node_weights2['real'], self.attn_params2['real_from_gene'])
        r_self = h_real @ self.node_weights2['real']
        h_real_new = self.dropout(F.relu(r_self + r_from_p + r_from_r + r_from_g + self.node_biases['real']))
        h_real_new = self.post(h_real, h_real_new)
    
        p_from_r = self.attn_second_layer(graph, 'real','pseudo', h_real,   h_pseudo, self.node_weights2['real'],  self.node_weights2['pseudo'], self.attn_params2['pseudo_from_real'])
        p_from_g = self.attn_second_layer(graph, 'gene','pseudo', h_gene,   h_pseudo, self.node_weights2['gene'],  self.node_weights2['pseudo'], self.attn_params2['pseudo_from_gene'])
        p_self = h_pseudo @ self.node_weights2['pseudo']
        h_pseudo_new = self.dropout(F.relu(p_self + p_from_r + p_from_g + self.node_biases['pseudo']))
        h_pseudo_new = self.post(h_pseudo, h_pseudo_new)
    
        g_from_p = self.attn_second_layer(graph, 'pseudo','gene', h_pseudo, h_gene, self.node_weights2['pseudo'], self.node_weights2['gene'], self.attn_params2['gene_from_pseudo'])
        g_from_r = self.attn_second_layer(graph, 'real',  'gene', h_real,   h_gene, self.node_weights2['real'],   self.node_weights2['gene'], self.attn_params2['gene_from_real'])
        g_self = h_gene @ self.node_weights2['gene']
        h_gene_new = self.dropout(F.relu(g_self + g_from_p + g_from_r + self.node_biases['gene']))
        h_gene_new = self.post(h_gene, h_gene_new)
        
        return h_real_new, h_pseudo_new, h_gene_new

    def forward(self, graph):
        h_real = graph.nodes['real'].data['feat']
        h_pseudo = graph.nodes['pseudo'].data['feat']
        h_gene = graph.nodes['gene'].data['feat']
        h_real, h_pseudo, h_gene = self.custom_update_layer1(graph, h_real, h_pseudo, h_gene)
        h_real, h_pseudo, h_gene = self.custom_update_layer2(graph, h_real, h_pseudo, h_gene)
        return self.pseudo_out(h_pseudo)

print('HeteroGAT模型定义完成')

HeteroGAT模型定义完成


In [7]:

real_spot_data = pd.read_csv('./real_spot_shared_genes.csv')
pseudo_spot_data = pd.read_csv('./pseudo_spot_shared_genes.csv')

real_spot_data = real_spot_data.drop(columns=["Unnamed: 0"]).values
pseudo_spot_data = pseudo_spot_data.drop(columns=["Unnamed: 0"]).values


gene_names = pd.read_csv('./real_spot_shared_genes.csv', index_col=0).columns.tolist()
disconnected_genes = []
with open('./disconnected_genes.txt', 'r') as f:
    for line in f:
        gene = line.strip()
        if gene:
            disconnected_genes.append(gene)

print(f"总基因数: {len(gene_names)}")
print(f"没有边连接的基因: {disconnected_genes}")
print(f"需要移除的基因数: {len(disconnected_genes)}")


disconnected_indices = [gene_names.index(gene) for gene in disconnected_genes if gene in gene_names]
print(f"移除的基因索引: {disconnected_indices}")

# 移除没有边连接的基因
if disconnected_indices:
    # 从基因表达矩阵中移除这些基因
    real_spot_data = np.delete(real_spot_data, disconnected_indices, axis=1)
    pseudo_spot_data = np.delete(pseudo_spot_data, disconnected_indices, axis=1)
    
    # 更新基因名称列表
    for idx in sorted(disconnected_indices, reverse=True):
        gene_names.pop(idx)
    
    print(f"移除后基因数: {len(gene_names)}")


adata_real = anndata.AnnData(real_spot_data)
adata_pseudo = anndata.AnnData(pseudo_spot_data)

sc.pp.log1p(adata_real)
sc.pp.log1p(adata_pseudo)

real_spot_feats = torch.tensor(adata_real.X, dtype=torch.float32).to(device)
pseudo_spot_feats = torch.tensor(adata_pseudo.X, dtype=torch.float32).to(device)
num_genes = real_spot_feats.shape[1]


real_spot_feats = zscore_norm(real_spot_feats)
pseudo_spot_feats = zscore_norm(pseudo_spot_feats)


all_spot_feats = torch.cat([real_spot_feats, pseudo_spot_feats], dim=0)
gene_by_spot = all_spot_feats.cpu().numpy().T
pca_dim = 16
pca = PCA(n_components=pca_dim)
gene_feats_pca = pca.fit_transform(gene_by_spot)
gene_feats = torch.tensor(gene_feats_pca, dtype=torch.float32, device=all_spot_feats.device)
gene_feats = zscore_norm(gene_feats)

print(f'Real spots: {real_spot_feats.shape[0]}, Pseudo spots: {pseudo_spot_feats.shape[0]}, Genes: {num_genes}')
print(f'Gene features shape: {gene_feats.shape}')

总基因数: 1300
没有边连接的基因: ['Gm42418', 'mt-Nd4']
需要移除的基因数: 2
移除的基因索引: [1190, 1298]
移除后基因数: 1298


C:\Users\lenovo\AppData\Local\Temp\ipykernel_10188\1890676944.py:38: FutureWarning: X.dtype being converted to np.float32 from float64. In the next version of anndata (0.9) conversion will not be automatic. Pass dtype explicitly to avoid this warning. Pass `AnnData(X, dtype=X.dtype, ...)` to get the future behavour.
  adata_real = anndata.AnnData(real_spot_data)
C:\Users\lenovo\AppData\Local\Temp\ipykernel_10188\1890676944.py:39: FutureWarning: X.dtype being converted to np.float32 from float64. In the next version of anndata (0.9) conversion will not be automatic. Pass dtype explicitly to avoid this warning. Pass `AnnData(X, dtype=X.dtype, ...)` to get the future behavour.
  adata_pseudo = anndata.AnnData(pseudo_spot_data)


Real spots: 2698, Pseudo spots: 5000, Genes: 1298
Gene features shape: torch.Size([1298, 16])


In [8]:

graph = create_heterogeneous_graph()
print(f"Graph nodes: real={graph.num_nodes('real')}, pseudo={graph.num_nodes('pseudo')}, gene={graph.num_nodes('gene')}")


pseudo_labels_df = pd.read_csv('./pseudo_spot_label_fractions.csv')
num_cell_types = pseudo_labels_df.shape[1]

print(f"细胞类型数: {num_cell_types}")
print(f"细胞类型: {pseudo_labels_df.columns.tolist()}")


in_dim_dict = {
    'real': real_spot_feats.shape[1], 
    'pseudo': pseudo_spot_feats.shape[1], 
    'gene': gene_feats.shape[1]
}

model = HeteroGAT(
    in_dim_dict=in_dim_dict, 
    hidden_dim=128, 
    num_cell_types=num_cell_types, 
    device=device, 
    num_heads=2, 
    dropout_rate=0.3
).to(device)


print("加载预训练模型...")
checkpoint = torch.load('hetero_gat_model800.pth', map_location=device)


if 'pseudo_out.weight' in checkpoint:
    checkpoint_num_classes = checkpoint['pseudo_out.weight'].shape[0]
    print(f"预训练模型输出维度: {checkpoint_num_classes}")
    print(f"当前模型输出维度: {num_cell_types}")
    
    if checkpoint_num_classes != num_cell_types:
        print(f"⚠️ 输出维度不匹配，跳过输出层参数加载")
        
        checkpoint.pop('pseudo_out.weight', None)
        checkpoint.pop('pseudo_out.bias', None)
        print("✓ 已移除输出层参数，将使用随机初始化")


model.load_state_dict(checkpoint, strict=False)
model.eval()
print('✓ 模型加载成功！')

Graph nodes: real=2698, pseudo=5000, gene=1298
细胞类型数: 8
细胞类型: ['Astrocytes', 'Blood', 'Ependymal', 'Excluded', 'Immune', 'Neurons', 'Oligos', 'Vascular']
加载预训练模型...
预训练模型输出维度: 8
当前模型输出维度: 8
✓ 模型加载成功！


In [9]:

print("📁 读取小鼠脑聚类标签...")


clusters_df = pd.read_csv('../data/clusters.csv', index_col=0)
print(f"  ✓ 聚类标签: {len(clusters_df)} spots")
print(f"  ✓ 聚类数: {clusters_df['Cluster'].nunique()}")
print(f"\n  聚类分布:")
for cluster, count in clusters_df['Cluster'].value_counts().sort_index().items():
    print(f"    • Cluster {cluster}: {count} spots")


clusters_df['Region'] = 'Cluster_' + clusters_df['Cluster'].astype(str)
print(f"\n  区域命名:")
for cluster in sorted(clusters_df['Cluster'].unique()):
    count = (clusters_df['Cluster'] == cluster).sum()
    print(f"    • Cluster {cluster} → Cluster_{cluster}: {count} spots")

print(f"\n✓ 小鼠脑区域标签加载完成！")

📁 读取小鼠脑聚类标签...
  ✓ 聚类标签: 2698 spots
  ✓ 聚类数: 9

  聚类分布:
    • Cluster 1: 370 spots
    • Cluster 2: 360 spots
    • Cluster 3: 352 spots
    • Cluster 4: 340 spots
    • Cluster 5: 295 spots
    • Cluster 6: 280 spots
    • Cluster 7: 255 spots
    • Cluster 8: 236 spots
    • Cluster 9: 210 spots

  区域命名:
    • Cluster 1 → Cluster_1: 370 spots
    • Cluster 2 → Cluster_2: 360 spots
    • Cluster 3 → Cluster_3: 352 spots
    • Cluster 4 → Cluster_4: 340 spots
    • Cluster 5 → Cluster_5: 295 spots
    • Cluster 6 → Cluster_6: 280 spots
    • Cluster 7 → Cluster_7: 255 spots
    • Cluster 8 → Cluster_8: 236 spots
    • Cluster 9 → Cluster_9: 210 spots

✓ 小鼠脑区域标签加载完成！


In [10]:
def extract_gene_importance_by_attention(model, graph, clusters_df):
    """基于模型的注意力机制提取基因重要性"""
    model.eval()
    
    # 使用全局的基因名称列表（已经移除了没有边连接的基因）
    regions = clusters_df['Cluster'].unique()
    
    region_gene_importance = {region: np.zeros(len(gene_names)) for region in regions}
    region_counts = {region: 0 for region in regions}
    
    # 获取real_spot的spot ID列表
    real_spot_df = pd.read_csv('./real_spot_shared_genes.csv', index_col=0)
    real_spot_ids = real_spot_df.index.tolist()
    
    print(f"Total real spots: {len(real_spot_ids)}")
    print(f"Spots with region labels: {len(clusters_df)}")
    
    with torch.no_grad():
        h_real = graph.nodes['real'].data['feat']
        h_pseudo = graph.nodes['pseudo'].data['feat']
        h_gene = graph.nodes['gene'].data['feat']
        
        for i in range(graph.num_nodes('real')):
            spot_id = real_spot_ids[i]
            
            if spot_id not in clusters_df.index:
                continue
            
            region = clusters_df.loc[spot_id, 'Cluster']
            
            gene_neighbors = graph.predecessors(i, etype=('gene', 'expressed_in_real', 'real'))
            
            if len(gene_neighbors) > 0:
                h_dst_i = h_real[i].unsqueeze(0).expand(len(gene_neighbors), -1)
                Wh_src = (h_gene[gene_neighbors] @ model.node_weights1['gene']).view(-1, model.num_heads, model.head_dim)
                
                v = torch.full((len(gene_neighbors),), i, dtype=gene_neighbors.dtype, device=gene_neighbors.device)
                eids = graph.edge_ids(gene_neighbors, v, etype=('gene', 'expressed_in_real', 'real'))
                edge_weights = graph.edges[('gene', 'expressed_in_real', 'real')].data['weight'][eids]
                
                attention_scores = torch.zeros(len(gene_neighbors), device=device)
                for head in range(model.num_heads):
                    Wh_src_head = Wh_src[:, head, :]
                    attn_input = torch.cat([h_dst_i, Wh_src_head], dim=1)
                    e_ij = F.leaky_relu(attn_input @ model.attn_params1['real_from_gene'][head].unsqueeze(-1)).squeeze(-1)
                    e_ij = e_ij + 0.1 * torch.log(edge_weights + 1e-8)
                    alpha = torch.softmax(e_ij, dim=0)
                    attention_scores += alpha
                
                attention_scores = attention_scores / model.num_heads
                
                gene_neighbors_cpu = gene_neighbors.cpu().numpy()
                attention_cpu = attention_scores.cpu().numpy()
                
                for gene_idx, attn_score in zip(gene_neighbors_cpu, attention_cpu):
                    if gene_idx < len(gene_names):
                        region_gene_importance[region][gene_idx] += attn_score
                
                region_counts[region] += 1
    
    results = {}
    print("\n" + "="*70)
    for region in regions:
        if region_counts[region] > 0:
            avg_importance = region_gene_importance[region] / region_counts[region]
            
            gene_df = pd.DataFrame({
                'gene': gene_names,
                'attention_score': avg_importance
            }).sort_values('attention_score', ascending=False)
            
            results[region] = gene_df
            # 修复：将region转换为字符串
            gene_df.to_csv(f'gene_attention_{str(region).replace(" ", "_")}.csv', index=False)

            print(f"\n{region} ({region_counts[region]} spots) - Top 10 重要基因:")
            print(gene_df[['gene', 'attention_score']].head(10).to_string(index=False))
    print("="*70)

    return results

In [11]:
# ==================== 差异表达分析函数 ====================

from scipy.stats import mannwhitneyu, ttest_ind

def differential_expression_analysis(df, region, gene_names, method='wilcoxon'):


    
    region_data = df[df['Region'] == region].drop(columns=['Region'])
    other_data = df[df['Region'] != region].drop(columns=['Region'])
    
    de_results = []
    
    for gene in gene_names:
        region_expr = region_data[gene].values
        other_expr = other_data[gene].values
        
        region_mean = region_expr.mean()
        other_mean = other_expr.mean()
        region_std = region_expr.std()
        other_std = other_expr.std()
        
        fc = np.log2((region_mean + 1e-8) / (other_mean + 1e-8))
        
        if method == 'wilcoxon':
            try:
                stat, pval = mannwhitneyu(region_expr, other_expr, alternative='two-sided')
            except:
                pval = 1.0
        else:
            try:
                stat, pval = ttest_ind(region_expr, other_expr)
            except:
                pval = 1.0
        
        pooled_std = np.sqrt(((len(region_expr)-1)*region_std**2 + (len(other_expr)-1)*other_std**2) / 
                            (len(region_expr) + len(other_expr) - 2))
        cohens_d = (region_mean - other_mean) / (pooled_std + 1e-8)
        
        de_results.append({
            'gene': gene,
            'log2FC': fc,
            'p_value': pval,
            'region_mean': region_mean,
            'other_mean': other_mean,
            'cohens_d': cohens_d,
            'region_expr_pct': (region_expr > 0).mean(),
            'other_expr_pct': (other_expr > 0).mean()
        })
    
    de_df = pd.DataFrame(de_results)
    
    from statsmodels.stats.multitest import multipletests
    _, pval_adj, _, _ = multipletests(de_df['p_value'], method='fdr_bh')
    de_df['p_adj'] = pval_adj
    
    de_df['significant'] = (de_df['p_adj'] < 0.05) & (abs(de_df['log2FC']) > 0.5)
    
    de_df['de_score'] = abs(de_df['log2FC']) * (-np.log10(de_df['p_adj'] + 1e-10))
    
    de_df = de_df.sort_values('de_score', ascending=False)
    
    return de_df

def perform_de_analysis_for_all_regions(clusters_df):
   
    from scipy import stats
    
    print("开始差异表达分析...")
    
    
    real_spot_df = pd.read_csv('./real_spot_shared_genes.csv', index_col=0)
    
    
    if disconnected_genes:
        real_spot_df = real_spot_df.drop(columns=disconnected_genes)
    
    # 合并聚类信息
    merged_df = real_spot_df.join(clusters_df['Cluster'])
    merged_df = merged_df.dropna(subset=['Cluster'])
    
    print(f"总spots: {len(merged_df)}")
    print(f"基因数: {len(gene_names)}")
    
    regions = clusters_df['Cluster'].unique()
    results = {}
    
    for region in regions:
        print(f"\n分析区域: {region}")
        
        
        region_data = merged_df[merged_df['Cluster'] == region].drop(columns=['Cluster'])
        other_data = merged_df[merged_df['Cluster'] != region].drop(columns=['Cluster'])
        
        if len(region_data) == 0 or len(other_data) == 0:
            print(f"  ⚠️ {region} 数据不足，跳过")
            continue
        
        de_results = []
        
        for gene in gene_names:
            if gene not in region_data.columns:
                continue
                
            
            region_expr = region_data[gene].values
            other_expr = other_data[gene].values
            
           
            region_mean = region_expr.mean()
            other_mean = other_expr.mean()
            
            if other_mean == 0:
                log2fc = np.log2(region_mean + 1e-8)
            else:
                log2fc = np.log2((region_mean + 1e-8) / (other_mean + 1e-8))
            
            # Wilcoxon
            try:
                stat, p_val = stats.mannwhitneyu(region_expr, other_expr, alternative='two-sided')
            except:
                p_val = 1.0
            
           
            de_score = abs(log2fc) * (-np.log10(p_val + 1e-8))
            
            de_results.append({
                'gene': gene,
                'log2FC': log2fc,
                'p_val': p_val,
                'p_adj': p_val,  
                'de_score': de_score
            })
        
        
        de_df = pd.DataFrame(de_results)
        de_df = de_df.sort_values('de_score', ascending=False)
        
        
        de_df.to_csv(f'DE_analysis_{str(region).replace(" ", "_")}.csv')

        
        significant_genes = de_df[de_df['p_adj'] < 0.05]

        results[region] = de_df

        print(f"  ✓ 保存到: DE_analysis_{str(region).replace(' ', '_')}.csv")
        print(f"  ✓ 显著基因: {len(significant_genes)}")
        print(f"  ✓ Top 5 基因: {de_df.head(5)['gene'].tolist()}")

    return results

In [13]:
print("="*100)
print("🔬 乳腺癌区域重要基因分析")
print("="*100)


print('\n1️⃣ 基于注意力的基因重要性分析...')
attention_results = extract_gene_importance_by_attention(model, graph, clusters_df)
print('\n✓ 注意力分析完成！')




print('\n3️⃣ 差异表达分析...')
de_results = perform_de_analysis_for_all_regions(clusters_df)
print('\n✓ DE分析完成！')

print("\n" + "="*100)
print("✅ 所有分析完成！结果已保存到CSV文件")
print("="*100)

🔬 乳腺癌区域重要基因分析

1️⃣ 基于注意力的基因重要性分析...
Total real spots: 2698
Spots with region labels: 2698


9 (210 spots) - Top 10 重要基因:
         gene  attention_score
          Ddn         0.003439
      Slc17a7         0.002924
      Bhlhe22         0.002750
          Nov         0.002747
        Nptxr         0.002677
      Slc30a3         0.002657
      Fam19a1         0.002649
       Plxnd1         0.002635
2010300C02Rik         0.002633
        Ccnd2         0.002624

5 (295 spots) - Top 10 重要基因:
         gene  attention_score
          Ddn         0.003879
       Camk2a         0.003416
      Fam107a         0.003178
       Pantr1         0.003123
         Hopx         0.002992
       Slc1a3         0.002917
        Mfge8         0.002755
       Cxcl14         0.002754
2900052N01Rik         0.002717
         Gja1         0.002683

8 (236 spots) - Top 10 重要基因:
   gene  attention_score
    Agt         0.003516
Slc6a11         0.003384
  Sparc         0.003278
  Zfhx3         0.003222
   Ache     